# Leccion 3: SVR — Modelo no-lineal para series temporales

**Dataset**: GEFCom2014 - 3 anios de carga electrica horaria

**Objetivo**: Construir un modelo SVR y compararlo con ARIMA.

---

## Que es SVR?

SVR = **S**upport **V**ector **R**egressor

Es una version de **SVM** (Support Vector Machine) para **regresion** (predecir valores continuos).

### Diferencia con ARIMA

| Caracteristica | ARIMA | SVR |
|----------------|-------|-----|
| Tipo | Lineal | No-lineal (con kernel) |
| Complejidad | Baja | Media |
| Velocidad | Lenta (walk-forward) | Rapida |
| Interpretabilidad | Alta | Baja |
| Uso comun | Series estacionarias | Cualquier serie |

### Cuando usar SVR?

- Cuando la serie tiene **no-linealidad** (ARIMA no captura bien)
- Cuando necesitas **rapidez** (SVR es mas rapido que ARIMA)
- Cuando quieres **comparar** modelos

## Analogia: Trazar una curva

Imagina que tienes puntos en 2D y quieres trazar una curva que pase cerca de todos:

- **ARIMA**: Dibuja una **linea recta** (modelo lineal)
- **SVR**: Dibuja una **curva flexible** (con kernel RBF)

Si los puntos siguen un patron curvo, SVR lo captura mejor. Si son lineales, ARIMA esta bien.

## 0. Instalar dependencias

In [ ]:
!pip install scikit-learn -q

## 1. Imports

| Libreria | Para que sirve |
|----------|----------------|
| `SVR` | El modelo Support Vector Regressor |
| `MinMaxScaler` | Escalar datos al rango [0,1] |
| `load_data` | Cargar el dataset |
| `mape` | Medir error de prediccion |

In [ ]:
import sys
sys.path.append('../../')

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt
import math

from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from common.utils import load_data, mape

warnings.filterwarnings('ignore')

print('Imports listos - SVR, MinMaxScaler, mape')

## 2. Cargar los datos

Mismo dataset que ARIMA: 26,304 horas de carga electrica.

In [ ]:
energy = load_data('../../data')[['load']]
print(f'Filas: {energy.shape[0]}')
print(f'Columnas: {list(energy.columns)}')
print(f'Rango: {energy.index.min()} -> {energy.index.max()}')
energy.head(5)

In [ ]:
energy.plot(y='load', subplots=True, figsize=(15, 8), fontsize=12)
plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.suptitle('Serie completa: Ene 2012 -> Dic 2014', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Split temporal

Misma logica que ARIMA: NUNCA random split.

```
Train: Nov 1 -> Dic 29, 2014  (1416 horas)
Test:  Dic 30 -> Dic 31, 2014 (48 horas)
```

In [ ]:
train_start_dt = '2014-11-01 00:00:00'
test_start_dt = '2014-12-30 00:00:00'

print(f'Train: {train_start_dt} -> {test_start_dt}')
print(f'Test:  {test_start_dt} -> 2014-12-31 23:00:00')

In [ ]:
# Visualizar train vs test
energy[(energy.index < test_start_dt) & (energy.index >= train_start_dt)][['load']]\
    .rename(columns={'load':'train'})\
    .join(energy[test_start_dt:][['load']].rename(columns={'load':'test'}), how='outer')\
    .plot(y=['train', 'test'], figsize=(15, 8), fontsize=12)

plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title('Train (naranja) vs Test (azul)', fontsize=14)
plt.legend(fontsize=12)
plt.show()

## 4. Escalar datos (MinMaxScaler)

Misma logica que ARIMA: `fit_transform()` en train, `transform()` en test.

In [ ]:
train = energy.copy()[(energy.index >= train_start_dt) & (energy.index < test_start_dt)][['load']]
test = energy.copy()[energy.index >= test_start_dt][['load']]

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

In [ ]:
# Escalar train
scaler = MinMaxScaler()
train['load'] = scaler.fit_transform(train)

print(f'Train min: {train["load"].min():.4f}')
print(f'Train max: {train["load"].max():.4f}')
train.head()

In [ ]:
# Escalar test
test['load'] = scaler.transform(test)
print(f'Test min: {test["load"].min():.4f}')
print(f'Test max: {test["load"].max():.4f}')
test.head()

## 5. Crear data con timesteps

### Que es timesteps?

SVR necesita que los datos tengan la forma **[batch, timesteps]**. Es decir, en vez de usar 1 valor para predecir, usamos **VARIOS valores pasados**.

Con `timesteps=5`:
```
Entrada: [t-4, t-3, t-2, t-1]  (4 valores pasados)
Salida:  [t]                    (1 valor a predecir)
```

### Por que 5 timesteps?

- **Muy pocos (2-3)**: El modelo no ve suficiente historial
- **5**: Buen balance (captura patron diario sin ser lento)
- **Muchos (10+)**: Mas datos, pero mas lento y puede sobreajustar

In [ ]:
# Converting to numpy arrays
train_data = train.values
test_data = test.values

# Seleccionar timesteps
timesteps = 5

print(f'Train data shape: {train_data.shape}')
print(f'Test data shape:  {test_data.shape}')

### Convertir a 2D tensor

Esta linea convierte la serie en ventanas de 5 valores:

```python
train_data_timesteps = np.array([[j for j in train_data[i:i+timesteps]] for i in range(0, len(train_data)-timesteps+1)])[:,:,0]
```

**Que hace?**
- Toma ventanas de 5 valores consecutivos
- Cada ventana se convierte en una fila
- El resultado es una matriz de (N-4) filas x 5 columnas

In [ ]:
# Convertir train a 2D tensor
train_data_timesteps = np.array([[j for j in train_data[i:i+timesteps]] for i in range(0, len(train_data)-timesteps+1)])[:,:,0]
print(f'Train timesteps shape: {train_data_timesteps.shape}')

# Ejemplo: primera ventana
print(f'\nEjemplo primera ventana (5 valores):')
print(train_data_timesteps[0])

In [ ]:
# Convertir test a 2D tensor
test_data_timesteps = np.array([[j for j in test_data[i:i+timesteps]] for i in range(0, len(test_data)-timesteps+1)])[:,:,0]
print(f'Test timesteps shape: {test_data_timesteps.shape}')

### Separar inputs y outputs

De cada ventana de 5 valores:
- **Inputs (X)**: los primeros 4 valores [t-4, t-3, t-2, t-1]
- **Output (y)**: el ultimo valor [t]

In [ ]:
# Separar inputs y outputs
x_train, y_train = train_data_timesteps[:,:timesteps-1], train_data_timesteps[:,[timesteps-1]]
x_test, y_test = test_data_timesteps[:,:timesteps-1], test_data_timesteps[:,[timesteps-1]]

print(f'x_train: {x_train.shape} (4 columnas = 4 valores pasados)')
print(f'y_train: {y_train.shape} (1 columna = valor a predecir)')
print(f'x_test:  {x_test.shape}')
print(f'y_test:  {y_test.shape}')

## 6. Crear modelo SVR

### Los 4 hiperparametros

| Parametro | Que controla | Nuestro valor |
|-----------|-------------|---------------|
| `kernel` | Tipo de curva | 'rbf' (Radial Basis Function) |
| `gamma` | Cuanto "influye" cada punto | 0.5 |
| `C` | Penalizacion por error | 10 |
| `epsilon` | Margen de tolerancia | 0.05 |

### Que es kernel RBF?

Es una funcion que transforma los datos a **mayor dimension** para encontrar patrones no-lineales.

```
Sin kernel: datos en linea recta
Con RBF:    datos en curva flexible
```

In [ ]:
# Crear modelo SVR
model = SVR(kernel='rbf', gamma=0.5, C=10, epsilon=0.05)

print(f'Kernel: {model.kernel}')
print(f'Gamma:  {model.gamma}')
print(f'C:      {model.C}')
print(f'Epsilon: {model.epsilon}')

In [ ]:
# Entrenar modelo
model.fit(x_train, y_train[:,0])

print('Modelo entrenado!')
print(f'Support vectors: {len(model.support_)}')

### Que son los support vectors?

Son los puntos de entrenamiento que **definen la frontera** del modelo. SVR usa solo estos puntos para predecir, no todos los datos.

- Pocos support vectors = modelo simple
- Muchos support vectors = modelo complejo

## 7. Hacer predicciones

In [ ]:
# Predecir
y_train_pred = model.predict(x_train).reshape(-1,1)
y_test_pred = model.predict(x_test).reshape(-1,1)

print(f'Predicciones train: {y_train_pred.shape}')
print(f'Predicciones test:  {y_test_pred.shape}')

## 8. Evaluar modelo

### Paso 1: Invertir escala

Las predicciones estan en [0,1]. Las convertimos a MW originales.

In [ ]:
# Invertir escala de predicciones
y_train_pred = scaler.inverse_transform(y_train_pred)
y_test_pred = scaler.inverse_transform(y_test_pred)

print(f'Predicciones train (MW): {y_train_pred[:3].flatten()}')
print(f'Predicciones test (MW):  {y_test_pred[:3].flatten()}')

In [ ]:
# Invertir escala de valores reales
y_train = scaler.inverse_transform(y_train)
y_test = scaler.inverse_transform(y_test)

print(f'Reales train (MW): {y_train[:3].flatten()}')
print(f'Reales test (MW):  {y_test[:3].flatten()}')

### Paso 2: Obtener timestamps para el grafico

Como usamos `timesteps-1` valores para la primera prediccion, los timestamps empiezan despues.

In [ ]:
# Obtener timestamps
train_timestamps = energy[(energy.index < test_start_dt) & (energy.index >= train_start_dt)].index[timesteps-1:]
test_timestamps = energy[test_start_dt:].index[timesteps-1:]

print(f'Train timestamps: {len(train_timestamps)}')
print(f'Test timestamps:  {len(test_timestamps)}')

### Paso 3: Graficar predicciones vs reales

In [ ]:
# Grafico de TRAIN
plt.figure(figsize=(25,6))
plt.plot(train_timestamps, y_train, color='red', linewidth=2.0, alpha=0.6, label='Actual')
plt.plot(train_timestamps, y_train_pred, color='blue', linewidth=0.8, label='Predicted')
plt.legend(fontsize=12)
plt.xlabel('Timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title('Train: Prediccion vs Real', fontsize=14)
plt.show()

In [ ]:
# MAPE train
train_mape = mape(y_train_pred, y_train) * 100
print(f'MAPE train: {train_mape:.2f}%')

In [ ]:
# Grafico de TEST
plt.figure(figsize=(10,3))
plt.plot(test_timestamps, y_test, color='red', linewidth=2.0, alpha=0.6, label='Actual')
plt.plot(test_timestamps, y_test_pred, color='blue', linewidth=0.8, label='Predicted')
plt.legend(fontsize=12)
plt.xlabel('Timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title('Test: Prediccion vs Real', fontsize=14)
plt.show()

In [ ]:
# MAPE test
test_mape = mape(y_test_pred, y_test) * 100
print(f'MAPE test: {test_mape:.2f}%')
print()
if test_mape < 2:
    print('Excelente: menos de 2% de error')
elif test_mape < 5:
    print('Bueno: menos de 5% de error')
else:
    print('Regular: mas de 5% de error')

## 9. Prediccion en el dataset completo

Para ver como se comporta el modelo en toda la serie.

In [ ]:
# Preparar datos completos
data = energy.copy().values
data = scaler.transform(data)

# Convertir a timesteps
data_timesteps = np.array([[j for j in data[i:i+timesteps]] for i in range(0, len(data)-timesteps+1)])[:,:,0]
print(f'Data timesteps shape: {data_timesteps.shape}')

# Separar X e Y
X, Y = data_timesteps[:,:timesteps-1], data_timesteps[:,[timesteps-1]]
print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')

In [ ]:
# Predecir
Y_pred = model.predict(X).reshape(-1,1)

# Invertir escala
Y_pred = scaler.inverse_transform(Y_pred)
Y = scaler.inverse_transform(Y)

In [ ]:
# Grafico completo
plt.figure(figsize=(30,8))
plt.plot(Y, color='red', linewidth=2.0, alpha=0.6, label='Actual')
plt.plot(Y_pred, color='blue', linewidth=0.8, label='Predicted')
plt.legend(fontsize=12)
plt.xlabel('Timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title('Dataset completo: Prediccion vs Real', fontsize=14)
plt.show()

In [ ]:
# MAPE completo
full_mape = mape(Y_pred, Y) * 100
print(f'MAPE completo: {full_mape:.2f}%')

## 10. Comparacion: SVR vs ARIMA

| Metrica | ARIMA | SVR |
|---------|-------|-----|
| MAPE test | ~0.5-1% | ~1-2% |
| Velocidad | Lenta (re-entrena) | Rapida |
| Complejidad | Media | Baja |
| Interpretabilidad | Alta (parametros claros) | Baja (caja negra) |

### Que modelo usar?

- **ARIMA**: Cuando la serie es estacionaria y quieres interpretabilidad
- **SVR**: Cuando la serie es no-lineal y quieres rapidez
- **Ambos**: Siempre es bueno comparar

## 11. Resumen

### Flujo completo

```
Cargar datos -> Split temporal -> Escalar -> Crear timesteps -> Entrenar SVR -> Predecir -> Evaluar
```

### Que aprendimos

| Concepto | Que es | Diferencia con ARIMA |
|----------|--------|----------------------|
| **SVR** | Support Vector Regressor | No-lineal vs lineal |
| **Kernel RBF** | Transforma datos a mayor dimension | ARIMA no usa kernels |
| **Timesteps** | Cuantos valores pasados mira | ARIMA usa lags directamente |
| **Support vectors** | Puntos que definen la frontera | ARIMA usa todos los datos |
| **Walk-forward** | No se usa en SVR | ARIMA lo necesita |

### Proximo paso

Comparar ARIMA y SVR en tu propio proyecto. Cual funciona mejor depende de tus datos.

---

**Fin de la seccion TimeSeries**. Buen trabajo!